# Market Microstructure Analysis

This notebook starts from captured aggregate trades and L5 feature bars, then plots trade volume, spread, midprice, and order-flow imbalance.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.storage import read_dataset

DATA_ROOT = ROOT / 'sample_data' / 'smoke'
trades = read_dataset(DATA_ROOT, 'raw_agg_trades')
features = read_dataset(DATA_ROOT, 'features')
trades['event_ts'] = pd.to_datetime(trades['event_ts'], utc=True)
features['bar_ts'] = pd.to_datetime(features['bar_ts'], utc=True)

In [ ]:
volume = trades.assign(bucket=trades['event_ts'].dt.floor('s')).groupby(['bucket', 'trade_side'])['quantity'].sum().unstack(fill_value=0)
volume.plot(kind='bar', stacked=True, figsize=(10, 4), title='Trade volume by side')
plt.ylabel('BTC quantity')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(features['bar_ts'], features['midprice'], label='midprice')
ax[0].set_title('Midprice')
ax[0].legend()
ax[1].plot(features['bar_ts'], features['spread'], label='spread')
ax[1].plot(features['bar_ts'], features['order_flow_imbalance'], label='order-flow imbalance')
ax[1].axhline(0, color='black', linewidth=0.8)
ax[1].legend()
plt.tight_layout()